In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import numpy as np
import cooler
from cooltools.lib.numutils import observed_over_expected, adaptive_coarsegrain
from cooltools.lib.numutils import interpolate_bad_singletons, set_diag, interp_nan
from astropy.convolution import Gaussian2DKernel
from astropy.convolution import convolve
from bioframe.io.fileops import read_bigwig

In [3]:
from cooltools.lib.numutils import zoom_array

In [4]:
import torch

In [5]:
# alpha genome - akita overlap table
df = pd.read_csv("/scratch1/smaruj/models_comparison_Akita_pytorch/alphagenome/human_cell_types/alphagenome_akita_orca_test_overlap.tsv", sep="\t")

In [6]:
# COOL_FILE = "/scratch1/smaruj/Akita_pytorch_training_data/human_unprocessed_data/Krietenstein2019_HFF/HiC_Krietenstein2019_HFF.hg38.mapq30.2048.cool"
COOL_FILE = "/scratch1/smaruj/Akita_pytorch_training_data/human_unprocessed_data/Krietenstein2019_H1hESC/HiC_Krietenstein2019_H1hESC.hg38.mapq30.2048.cool"

genome_hic_cool = cooler.Cooler(COOL_FILE)

In [7]:
import math

def from_upper_triu(vecs, matrix_len=None, symmetric=True):
    """
    Reconstruct a square matrix (or batch of matrices) from its vectorized upper triangle.

    Parameters
    ----------
    vecs : torch.Tensor
        Tensor of shape (N,) or (B, N), where N = L*(L+1)/2 and L is the matrix side length.
    matrix_len : int, optional
        Size of the matrix (L). If None, it will be inferred automatically.
    symmetric : bool, default=True
        If True, fills the lower triangle to make the matrix symmetric.

    Returns
    -------
    mats : torch.Tensor
        Tensor of shape (L, L) if vecs is 1D, else (B, L, L).
    """
    if vecs.ndim == 1:
        vecs = vecs.unsqueeze(0)  # add batch dimension
        single = True
    else:
        single = False

    B, N = vecs.shape

    # Infer matrix dimension if not provided
    if matrix_len is None:
        matrix_len = int((math.sqrt(8 * N + 1) - 1) / 2)
        if matrix_len * (matrix_len + 1) // 2 != N:
            raise ValueError(f"Vector length {N} does not correspond to a valid triangular number.")

    triu_indices = torch.triu_indices(matrix_len, matrix_len, device=vecs.device)
    mats = torch.zeros((B, matrix_len, matrix_len), device=vecs.device, dtype=vecs.dtype)
    mats[:, triu_indices[0], triu_indices[1]] = vecs

    if symmetric:
        mats = mats + mats.transpose(1, 2) - torch.diag_embed(torch.diagonal(mats, dim1=1, dim2=2))

    if single:
        mats = mats.squeeze(0)  # remove batch dimension

    return mats


In [8]:
import re

def extract_coordinates_from_mseq(mseq_str):
    # Regular expression to match the format: chrom:start-end
    match = re.match(r"(?P<chrom>\w+):(?P<start>\d+)-(?P<end>\d+)", mseq_str)
    
    if match:
        chrom = match.group('chrom')
        start = int(match.group('start'))
        end = int(match.group('end'))
        return chrom, start, end
    else:
        raise ValueError(f"Invalid mseq_str format: {mseq_str}")

In [9]:
def process_hic_matrix(genome_hic_cool, mseq_str, diagonal_offset=2, padding=64, kernel_stddev=1, bin_size=2048, gaps_df=None):
    seq_hic_raw = genome_hic_cool.matrix(balance=True).fetch(mseq_str)
    
    chrom, start, end = extract_coordinates_from_mseq(mseq_str)
    
    # Check for NaN filtering percentage
    seq_hic_nan = np.isnan(seq_hic_raw)
    num_filtered_bins = np.sum(np.sum(seq_hic_nan, axis=0) == len(seq_hic_nan))
    print("num_filtered_bins:", num_filtered_bins)
    
    if num_filtered_bins > (0.5 * len(seq_hic_nan)):
        print(f"More than 50% bins filtered in {mseq_str}. Check Hi-C data quality.")
    
    ###########
    # Mask for rows/columns full of NaNs
    row_nan_mask = np.all(seq_hic_nan, axis=1)  # Rows with all NaNs
    col_nan_mask = np.all(seq_hic_nan, axis=0)  # Columns with all NaNs
    
    true_row_indices = np.where(row_nan_mask)[0]
    print(f"Indices of rows with NaNs: {true_row_indices}")
    
    # Apply the NaN mask earlier in the process to avoid processing NaN-only rows/columns
    seq_hic_raw[row_nan_mask, :] = np.nan  # Mask entire rows
    seq_hic_raw[:, col_nan_mask] = np.nan  # Mask entire columns
    
    # Check for NaN filtering percentage
    num_filtered_bins = np.sum(np.sum(seq_hic_nan, axis=0) == len(seq_hic_nan))
    print("num_filtered_bins:", num_filtered_bins)
    ###########
    
    # Mask for regions overlapping with gaps
    if gaps_df is not None:
        # Filter gaps_df for the current chromosome
        gaps_chr = gaps_df[gaps_df['chrom'] == chrom]
        
        # Iterate through each gap region and mark the corresponding rows and columns as NaN
        for _, gap in gaps_chr.iterrows():
            gap_start = gap['start']
            gap_end = gap['end']
            
            # Check if the gap overlaps with the current region
            if (gap_start < end) and (gap_end > start):
                # Mark rows and columns that fall within the gap range as NaN
                gap_start_idx = max(gap_start - start, 0) // bin_size  # Avoid negative indices
                gap_end_idx = min(gap_end - start, seq_hic_raw.shape[0]) // bin_size # Avoid out of bounds
                
                # Add the affected rows and columns to the NaN mask
                row_nan_mask[gap_start_idx:gap_end_idx] = True
                col_nan_mask[gap_start_idx:gap_end_idx] = True
                
        # Apply the updated NaN mask for gaps
        seq_hic_raw[row_nan_mask, :] = np.nan
        seq_hic_raw[:, col_nan_mask] = np.nan
    
        true_row_indices = np.where(row_nan_mask)[0]
        print(f"Indices of rows with NaNs: {true_row_indices}")
    
    # clip first diagonals and high values
    clipval = np.nanmedian(np.diag(seq_hic_raw, diagonal_offset))
    for i in range(-diagonal_offset+1, diagonal_offset):
        set_diag(seq_hic_raw, clipval, i)
    seq_hic_raw = np.clip(seq_hic_raw, 0, clipval)
    seq_hic_raw[seq_hic_nan] = np.nan
    
    # adaptively coarsegrain based on raw counts
    seq_hic_smoothed = adaptive_coarsegrain(
                            seq_hic_raw,
                            genome_hic_cool.matrix(balance=False).fetch(mseq_str),
                            cutoff=2, max_levels=8)
    seq_hic_nan = np.isnan(seq_hic_smoothed)
    
    # local obs/exp
    seq_hic_obsexp = observed_over_expected(seq_hic_smoothed, ~seq_hic_nan)[0]
    
    log_hic_obsexp = np.log(seq_hic_obsexp)
    
    # Apply padding
    if padding > 0:
        log_hic_obsexp = log_hic_obsexp[padding:-padding, padding:-padding]
        row_nan_mask = row_nan_mask[padding:-padding]
        col_nan_mask = col_nan_mask[padding:-padding]
        
    log_hic_obsexp = interp_nan(log_hic_obsexp)
    for i in range(-diagonal_offset+1, diagonal_offset): set_diag(log_hic_obsexp, 0,i)
    
    kernel = Gaussian2DKernel(x_stddev=kernel_stddev)
    seq_hic = convolve(log_hic_obsexp, kernel)
    
    return seq_hic

In [10]:
def upper_triangular_to_vector_skip_diagonals(matrix, dim=512, diag=2):
    
    # Extract the upper triangular part excluding the first two diagonals
    upper_triangular_vector = matrix[np.triu_indices(dim, k=diag)]
    
    return upper_triangular_vector

In [11]:
N = 256
diagonal_offset = 2

In [12]:
import matplotlib.pyplot as plt
import seaborn as sns

In [13]:
def plot_map(matrix, vmin=-0.6, vmax=0.6, palette="RdBu_r", width=5, height=5):
    fig, axes = plt.subplots(1, 1, figsize=(width, height))

    sns.heatmap(
        matrix,
        vmin=vmin,
        vmax=vmax,
        cbar=True,
        cmap=palette,
        square=True,
        xticklabels=False,
        yticklabels=False,
        ax=axes
    )

    plt.tight_layout()
    plt.show()

In [14]:
gap_df_path = "/project2/fudenber_735/backup/DNN_HiC/data_hg38/hg38.blacklist.rep.bed"
gap_df = pd.read_csv(gap_df_path, sep="\t", header=None, names=["chrom", "start", "end", "fold"])

In [15]:
# Exclude diagonals: 0 and ±1
def get_upper_tri_mask(n, skip_diagonals=2):
    # Create mask with False on excluded diagonals, True elsewhere in upper triangle
    mask = np.triu(np.ones((n, n), dtype=bool), k=skip_diagonals)
    return mask

In [16]:
# data_path = "/scratch1/smaruj/orca_validation/human_hff_4kb_preds"
data_path = "/scratch1/smaruj/orca_validation/human_H1hESC_4kb_preds"

In [17]:
all_preds = []
all_targets = []

for i, row in enumerate(df.itertuples(index=False)):
    print("index:", i)
    chr, start, end = row.chr, row.start, row.end
    mseq_str = f"{chr}:{start}-{end}"
    
    matrix = process_hic_matrix(genome_hic_cool, mseq_str, diagonal_offset=2, padding=64, kernel_stddev=1, bin_size=2048, gaps_df=gap_df)
    
    # print("target")
    # plot_map(matrix)
    
    mask = get_upper_tri_mask(matrix.shape[0])
    target_vec = matrix[mask]
    
    # print(target_vec.shape)
    
    # ORCA PRED
    # Load the PyTorch tensor
    tensor = torch.load(f"{data_path}/{chr}_{start}_{end}_orca_pred.pt", map_location='cpu')
    
    numpy_array = from_upper_triu(tensor, 262).cpu().numpy()
    
    # print("pred, 262x262")
    # plot_map(numpy_array)
    
    interpolated_matrix = zoom_array(in_array = numpy_array,
                                final_shape=(512, 512))
    
    pred_vec = interpolated_matrix[mask]
    
    # print(pred_vec.shape)
    
    # print("inter pred, 512x512")
    # plot_map(interpolated_matrix)
    
    all_targets.append(target_vec)
    all_preds.append(pred_vec)
    

index: 0
num_filtered_bins: 25
Indices of rows with NaNs: [  1   2   3   4   5   6   8   9  10  11  34 133 172 203 335 341 573 574
 579 580 584 591 599 612 613]
num_filtered_bins: 25
Indices of rows with NaNs: [  1   2   3   4   5   6   8   9  10  11  34 133 172 203 335 341 573 574
 579 580 584 591 599 612 613]


/home1/smaruj/miniconda3/envs/pytorch_cuda11.8/lib/python3.11/site-packages/cooltools/lib/numutils.py:1376: RuntimeWarning: invalid value encountered in divide
  val_cur = ar_cur / armask_cur


index: 1
num_filtered_bins: 13
Indices of rows with NaNs: [ 12  43 175 181 413 414 419 420 424 431 439 452 453]
num_filtered_bins: 13
Indices of rows with NaNs: [ 12  43 175 181 413 414 419 420 424 431 439 452 453]


/home1/smaruj/miniconda3/envs/pytorch_cuda11.8/lib/python3.11/site-packages/cooltools/lib/numutils.py:1376: RuntimeWarning: invalid value encountered in divide
  val_cur = ar_cur / armask_cur


index: 2
num_filtered_bins: 13
Indices of rows with NaNs: [ 15  21 253 254 259 260 264 271 279 292 293 486 519]
num_filtered_bins: 13
Indices of rows with NaNs: [ 15  21 253 254 259 260 264 271 279 292 293 486 519]
index: 3
num_filtered_bins: 11
Indices of rows with NaNs: [ 93  94  99 100 104 111 119 132 133 326 359]
num_filtered_bins: 11
Indices of rows with NaNs: [ 93  94  99 100 104 111 119 132 133 326 359]
index: 4
num_filtered_bins: 3
Indices of rows with NaNs: [166 199 567]
num_filtered_bins: 3
Indices of rows with NaNs: [166 199 567]
index: 5
num_filtered_bins: 3
Indices of rows with NaNs: [  6  39 407]
num_filtered_bins: 3
Indices of rows with NaNs: [  6  39 407]
index: 6
num_filtered_bins: 1
Indices of rows with NaNs: [247]
num_filtered_bins: 1
Indices of rows with NaNs: [247]
index: 7
num_filtered_bins: 1
Indices of rows with NaNs: [87]
num_filtered_bins: 1
Indices of rows with NaNs: [87]
index: 8
num_filtered_bins: 0
Indices of rows with NaNs: []
num_filtered_bins: 0
Indices

In [18]:
from scipy.stats import spearmanr, pearsonr

In [19]:
all_cropped_preds = np.array(all_preds)
all_targets = np.array(all_targets)

In [20]:
# Flatten the arrays
preds_flat = all_cropped_preds.flatten()
targets_flat = all_targets.flatten()

# Create a mask for valid (non-NaN) entries
valid_mask = ~np.isnan(preds_flat) & ~np.isnan(targets_flat)

# Apply mask
preds_valid = preds_flat[valid_mask]
targets_valid = targets_flat[valid_mask]

In [21]:
# Compute metrics
pearR = pearsonr(preds_valid, targets_valid)[0]
spearmanR = spearmanr(preds_valid, targets_valid)[0]
mse = np.mean((targets_valid - preds_valid) ** 2)

In [22]:
print("Pearson R = ", pearR)
print("Spearnman R = ", spearmanR)
print("MSE = ", mse)

Pearson R =  0.6643121904631598
Spearnman R =  0.6273403611386141
MSE =  0.14424705921919856


In [23]:
print("Pearson R = ", pearR)
print("Spearnman R = ", spearmanR)
print("MSE = ", mse)

Pearson R =  0.6643121904631598
Spearnman R =  0.6273403611386141
MSE =  0.14424705921919856
